# customer-dp : pipeline Bronze → Silver → Gold pas à pas

**Scénario** : des commandes et des clients (CSV bruts) sont ingérés, nettoyés, dédupliqués,
enrichis puis agrégés par jour.

```text
CSV bruts → Bronze (traçabilité) → Silver (nettoyage, dédup, jointure) → Gold (agrégats quotidiens)
```

**Objectifs** : comprendre chaque étape en l'exécutant, sans réécrire la logique : le notebook
**réutilise** le code du package `customer_dp` (configuration, session Spark, transformations, contrôles).

**Mode d'exécution** : par défaut *local* (kernel du `.venv` uv, Spark local, CSV d'exemple, sorties
temporaires). Pour un autre environnement, exportez les variables `CUSTOMER_DP_*` du profil avant de
lancer le kernel (voir README). Un kernel local ne réutilise **pas** une session Spark distante :
la validation sur YARN se fait avec `scripts/run.sh` / `scripts/test_cluster.sh`.

## 1. Initialisation et configuration partagée

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="customer-dp-nb-"))

# Valeurs locales par défaut ; des variables déjà exportées (autre environnement) restent prioritaires.
os.environ.setdefault("CUSTOMER_DP_INPUT_ROOT", f"file://{ROOT / 'data' / 'sample'}")
os.environ.setdefault("CUSTOMER_DP_OUTPUT_ROOT", f"file://{OUTPUT_DIR}")
os.environ.setdefault("CUSTOMER_DP_SPARK_MASTER", "local[2]")

from customer_dp.common.config import load_config  # noqa: E402

config = load_config()
print("Entrées :", config.input_root)
print("Sorties :", config.output_root)

## 2. SparkSession partagée

`get_spark` crée la session ou récupère la session active : on n'en crée pas une par cellule.

In [ ]:
from customer_dp.common.spark import get_spark  # noqa: E402

spark = get_spark(config, "customer-dp-notebook")
spark.sparkContext.setLogLevel("WARN")
spark.version

## 3. Lecture et exploration des fichiers bruts

Les schémas sont **explicites** (tout en chaînes) : pas d'inférence, et les valeurs invalides
restent visibles pour être qualifiées en Silver.

In [ ]:
from customer_dp.bronze.job import CUSTOMERS_SCHEMA, ORDERS_SCHEMA, add_ingestion_metadata, read_csv  # noqa: E402

raw_orders = read_csv(spark, config.input_path("orders"), ORDERS_SCHEMA)
raw_customers = read_csv(spark, config.input_path("customers"), CUSTOMERS_SCHEMA)
print("commandes brutes :", raw_orders.count(), "| clients :", raw_customers.count())
raw_orders.show(8, truncate=False)

## 4. Bronze : traçabilité

Ajout de `ingestion_timestamp`, `source_file` et `batch_date`, puis écriture Parquet partitionnée par lot.

In [ ]:
from customer_dp.common.io import read_table, write_table  # noqa: E402

BATCH_DATE = "2026-03-10"
bronze_orders = add_ingestion_metadata(raw_orders, BATCH_DATE)
bronze_customers = add_ingestion_metadata(raw_customers, BATCH_DATE)
write_table(bronze_orders, config.table("bronze_orders"), config, spark)
write_table(bronze_customers, config.table("bronze_customers"), config, spark)
read_table(spark, config.table("bronze_orders")).select("order_id", "source_file", "batch_date").show(3, truncate=60)

## 5. Silver : nettoyage

Normalisation des chaînes (trim, majuscules), chaînes vides → `NULL`, conversion des types
(`date`, `timestamp`, `decimal(12,2)`). Une conversion impossible donne `NULL` : elle est qualifiée à l'étape suivante.

In [ ]:
from customer_dp.silver.job import (  # noqa: E402
    add_event_date,
    clean_customers,
    clean_orders,
    deduplicate_latest,
    enrich_with_customers,
    split_valid_invalid,
)

cleaned = clean_orders(read_table(spark, config.table("bronze_orders")))
cleaned.printSchema()
cleaned.show(5, truncate=False)

## 6. Validation et rejets

Les commandes invalides ne sont pas perdues : elles vont dans une table de rejets avec leur raison.

In [ ]:
valid, rejected = split_valid_invalid(cleaned, config.valid_statuses)
rejected.select("order_id", "amount", "status", "reject_reason").show(truncate=False)

## 7. Déduplication par fenêtre Spark

`row_number()` sur `order_id`, trié par `updated_at` décroissant puis critères de départage fixes :
le résultat est **déterministe**. Exemple : `O1001` a deux versions (NEW puis PAID), `O1012` deux
versions au même `updated_at`.

In [ ]:
deduped = deduplicate_latest(valid)
deduped.filter("order_id IN ('O1001', 'O1012')").select("order_id", "status", "amount", "updated_at").show()

## 8. Jointure avec les clients

La table clients est petite : **broadcast join** (pas de shuffle des commandes). Les clients inconnus
sont conservés et signalés par `is_known_customer`.

In [ ]:
customers = clean_customers(read_table(spark, config.table("bronze_customers")))
silver = enrich_with_customers(add_event_date(deduped), customers, config.broadcast_customers)
silver.select(
    "order_id", "customer_id", "customer_name", "country", "is_known_customer", "event_date"
).show(truncate=False)

## 9. Plan d'exécution

Cherchez `BroadcastHashJoin` et la fenêtre (`Window`) dans le plan physique.

In [ ]:
silver.explain()

## 10. Partitionnement et écriture Silver

Partitionnement par `event_date` (faible cardinalité, filtre fréquent) : un dossier par jour.

In [ ]:
write_table(silver, config.table("silver_orders"), config, spark)
write_table(rejected, config.table("silver_orders_rejected"), config, spark)
silver_path = OUTPUT_DIR / "silver" / "silver_orders"
print(sorted(p.name for p in silver_path.glob("event_date=*")) if silver_path.exists() else "(sorties distantes)")

## 11. Gold : agrégation quotidienne

**Granularité** : une ligne par `event_date`. Seuls les statuts `PAID` et `SHIPPED` sont comptés.
**Types** : montants en `decimal(18,2)` (jamais de `float` pour de l'argent) ; moyenne arrondie
`HALF_UP` à 2 décimales.

In [ ]:
from customer_dp.gold.job import daily_sales  # noqa: E402

gold = daily_sales(read_table(spark, config.table("silver_orders")), config.counted_statuses, config.amount_scale)
write_table(gold, config.table("gold_daily_sales"), config, spark)
read_table(spark, config.table("gold_daily_sales")).orderBy("event_date").show()

## 12. Contrôles de qualité (partagés avec le job `verify`)

In [ ]:
from customer_dp.common.quality import assert_all_passed, check_gold, check_row_counts, check_silver  # noqa: E402

silver_df = read_table(spark, config.table("silver_orders"))
gold_df = read_table(spark, config.table("gold_daily_sales"))
results = (
    check_silver(silver_df)
    + check_gold(silver_df, gold_df, config.counted_statuses)
    + check_row_counts(
        {"silver_rows": silver_df.count(), "rejected_rows": rejected.count(), "gold_rows": gold_df.count()},
        {"silver_rows": 9, "rejected_rows": 5, "gold_rows": 3},
    )
)
for r in results:
    print("OK  " if r.passed else "FAIL", r.name, r.detail)
assert_all_passed(results)

## 13. Cache, `persist()` et `unpersist()`

Un DataFrame **coûteux et réutilisé** (ici `silver`, issu d'une fenêtre et d'une jointure) est calculé
plusieurs fois par chaque action. Cycle : calcul → `persist()` → action qui matérialise → réutilisations
→ `unpersist()`, garanti par `try/finally` même en cas d'erreur.

- `cache()`/`persist()` évitent des **recalculs** mais **ne tronquent pas le lineage** logique.
- `checkpoint()` est un mécanisme distinct : il écrit sur disque et coupe le lineage. À réserver aux
  plans très longs (itérations, lineage énorme) car il a un coût d'écriture ; inutile ici.
- Ne pas mettre en cache tous les DataFrames intermédiaires : un DataFrame utilisé une seule fois n'y gagne rien.

In [ ]:
from contextlib import contextmanager  # noqa: E402

from pyspark import StorageLevel  # noqa: E402


@contextmanager
def persisted(df, level=StorageLevel.MEMORY_AND_DISK):
    df.persist(level)
    try:
        yield df
    finally:
        df.unpersist()


with persisted(silver) as reused:
    total = reused.count()  # action : matérialise le cache
    unknown = reused.filter("NOT is_known_customer").count()  # réutilise le cache
    per_day = reused.groupBy("event_date").count().collect()  # petit résultat : collect() sans risque
    by_day = {str(r.event_date): r["count"] for r in per_day}
    print("is_cached pendant le bloc :", reused.is_cached)
print(f"lignes={total} clients_inconnus={unknown} par_jour={by_day}")
print("is_cached après le bloc :", silver.is_cached)

## 14. Conclusion et nettoyage

- Chaque couche est exécutable seule : `scripts/run.sh --layer bronze|silver|gold`.
- Le nettoyage ci-dessous ne **stoppe pas** la session Spark (elle peut être partagée) : il vide les
  caches et supprime les sorties temporaires du notebook.

Prochaines étapes : packager (`scripts/package.sh`), publier (`scripts/publish_hdfs.sh`), tester sur le
cluster (`scripts/test_cluster.sh`), puis déployer le DAG (`scripts/deploy_dag.sh`).

In [ ]:
spark.catalog.clearCache()
if str(OUTPUT_DIR).startswith(tempfile.gettempdir()):
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)